# Geometry — defining the walkable area

In JuPedSim, the **walkable area** is a [Shapely](https://shapely.readthedocs.io) polygon.
Holes in the polygon become **obstacles** (walls, pillars, furniture).
Agents are automatically routed around them by the internal navigation mesh.

This notebook shows how to:
1. Create a room with a square pillar obstacle.
2. Run a small simulation so agents navigate around the pillar.
3. Visualise the trajectories.

See also: {doc}`Getting started </notebooks/fundamentals/00_getting_started>` · {doc}`Exit Stage </notebooks/fundamentals/03_exits>`

In [ ]:
import matplotlib.pyplot as plt
import shapely
from pedpy import WalkableArea, plot_walkable_area

# A 10 x 10 room with a square pillar (a hole) in the middle.
geometry = shapely.Polygon(
    [(0, 0), (10, 0), (10, 10), (0, 10)],
    holes=[[(4, 4), (6, 4), (6, 6), (4, 6)]],
)
plot_walkable_area(walkable_area=WalkableArea(geometry)).set_aspect("equal")
plt.show()

In [ ]:
import pathlib

import jupedsim as jps

trajectory_file = pathlib.Path("geometry.sqlite")

simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=geometry,
    trajectory_writer=jps.SqliteTrajectoryWriter(output_file=trajectory_file),
)

# Single exit on the right side of the room.
exit_id = simulation.add_exit_stage([(9, 4), (10, 4), (10, 6), (9, 6)])
journey_id = simulation.add_journey(jps.JourneyDescription([exit_id]))

# Place 20 agents on the left side (avoiding the pillar at x=4–6, y=4–6).
n_agents = 20
positions = jps.distributions.distribute_by_number(
    polygon=shapely.Polygon([(0.5, 0.5), (3, 0.5), (3, 9.5), (0.5, 9.5)]),
    number_of_agents=n_agents,
    distance_to_agents=0.4,
    distance_to_polygon=0.2,
    seed=1,
)
for position in positions:
    simulation.add_agent(
        jps.CollisionFreeSpeedModelAgentParameters(
            journey_id=journey_id,
            stage_id=exit_id,
            position=position,
            radius=0.12,
        )
    )

while simulation.agent_count() > 0 and simulation.iteration_count() < 10_000:
    simulation.iterate()

print(
    f"Evacuated {n_agents} agents in {simulation.iteration_count()} iterations "
    f"({simulation.elapsed_time():.1f} s)."
)

In [ ]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data, walkable_area = read_sqlite_file(trajectory_file)
animate(trajectory_data, walkable_area)

## Try one change

Add a second pillar by extending the `holes` list:

```python
geometry = shapely.Polygon(
    [(0, 0), (10, 0), (10, 10), (0, 10)],
    holes=[
        [(4, 4), (6, 4), (6, 6), (4, 6)],   # first pillar
        [(7, 1), (8, 1), (8, 2), (7, 2)],   # second pillar
    ],
)
```

Re-run the cells above and observe how agents route around both obstacles.

## See also

- {doc}`Getting started </notebooks/fundamentals/00_getting_started>` — first simulation from scratch.
- {doc}`Exit Stage </notebooks/fundamentals/03_exits>` — multiple exits and exit selection.
- [JuPedSim scenarios](https://scenarios.jupedsim.org) — more complex geometries.